# RDKitとW&Bで化合物を分析する

第7章 応用事例のサンプルです。上から順に実行してください。

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nejumi/wandb_fun_book_public/blob/main/notebooks/rdkit_chemical_analysis.ipynb)

保存先は接続設定の `ENTITY`（team名）と `PROJECT` で指定できます。APIキーは `wandb.login()` の画面から入力してください。

In [ ]:
%pip install -q "wandb>=0.31,<0.32" "xgboost>=3.4,<4" rdkit pandas scikit-learn pillow tqdm

# ライブラリのインポート

In [ ]:
import numpy as np
import pandas as pd
from pandas import DataFrame
from sklearn.model_selection import train_test_split
from rdkit import Chem
from rdkit.Chem import AllChem, Draw
import xgboost as xgb
from xgboost import XGBRegressor
from rdkit.Chem.Draw import rdMolDraw2D
from IPython.display import Image
from wandb.integration.xgboost import WandbCallback

from tqdm import tqdm
tqdm.pandas()

import wandb
from PIL import Image as PILImage
from io import BytesIO


# W&Bへのログインとランの初期化など準備

In [ ]:
import os
import wandb

ENTITY = os.environ.get("WANDB_ENTITY", "")  # 保存先のteam名。空欄なら既定値を使用
PROJECT = os.environ.get("WANDB_PROJECT", "book-v4-chemical-analysis")
if ENTITY:
    os.environ["WANDB_ENTITY"] = ENTITY
wandb.login()


In [ ]:
run = wandb.init(project=PROJECT)

# データを読み込んで特徴抽出を行う

In [ ]:
# データの読み込み
file_path = wandb.use_artifact('yuya-yamamoto/save_data_artifact/dataset_filtered:v0', type='dataset').download()
data = pd.read_csv(file_path + '/drugbank_smiles_filtered.csv')

smiles_col = "smiles"
target_col = "logP ALOGPS"

In [ ]:
data

In [ ]:
# ここでは簡単のために大きすぎる分子はラフに除外しておく
data = data[data.smiles.apply(len)<100]

In [ ]:
# WandbCallbackを継承して、feature importanceのテーブルにビットの可視化を追加する
class WandbCallbackWithStructures(WandbCallback):
    def __init__(self, first_observed_fragments, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.first_observed_fragments = first_observed_fragments

    def _log_feature_importance(self, model):
        fi = model.get_score(importance_type=self.importance_type)
        fi_data = [[k, fi[k], wandb.Image(self.first_observed_fragments[int(k.split("_")[-1])])] for k in fi]
        table = wandb.Table(data=fi_data, columns=["Feature", "Importance", "Fragment"])
        wandb.log(
            {
                "Feature Importance": wandb.plot.bar(
                    table, "Feature", "Importance", title="Feature Importance"
                )
            }
        )

In [ ]:
# NaNの行を削除
data = data[data[target_col ].isnull()==False]

# データをトレーニングセットとバリデーションセットに分割
train, validation = train_test_split(data, test_size=0.025, random_state=71)

# SMILESからECFP4記述子、molオブジェクト、ビット情報を抽出する関数
nBits = 2048
r = 2
def smiles_to_ecfp4_and_mol_and_bit_info(smiles):
    mol = Chem.MolFromSmiles(smiles)
    bit_info = {}
    ecfp4 = AllChem.GetMorganFingerprintAsBitVect(mol, r, nBits=nBits, bitInfo=bit_info)
    return np.array(ecfp4), mol, bit_info

# ECFP4記述子、molオブジェクト、ビット情報を抽出
X_train, mols_train, bit_info_train = zip(*[smiles_to_ecfp4_and_mol_and_bit_info(smiles) for smiles in train[smiles_col]])
X_val, mols_val, bit_info_val = zip(*[smiles_to_ecfp4_and_mol_and_bit_info(smiles) for smiles in validation[smiles_col]])

# DataFrameに変換
col_name = [f'ECFP_{str(i).zfill(len(str(nBits)))}' for i in range(nBits)]
X_train = DataFrame(X_train, index=range(len(X_train)), columns=col_name)
X_val = DataFrame(X_val, index=range(len(X_train), len(X_train)+len(X_val)), columns=col_name)

# 目的変数を抽出
y_train = train[target_col].values
y_val = validation[target_col].values

In [ ]:
# ここでは一定のインサイトを得るための工学的判断としてハッシュ衝突は無視して最初に観測したビットを記録する
first_observed_fragments = {}

for bit_info, mol in tqdm(zip(bit_info_train, mols_train)):
    for bit, info in bit_info.items():
        if bit not in first_observed_fragments:
            fragment = Draw.DrawMorganBit(mol, bit, bit_info, useSVG=False)
            if isinstance(fragment, bytes):
                fragment = PILImage.open(BytesIO(fragment))
            first_observed_fragments[bit] = fragment

first_observed_fragments = dict(sorted(first_observed_fragments.items()))

# 簡単にモデルの学習を行う

In [ ]:
# XGBoostでモデルを学習
model = XGBRegressor(
    max_depth=8,
    learning_rate=0.1,
    n_estimators=1500,
    tree_method='hist',
    eval_metric='rmse',
    early_stopping_rounds=20,
    callbacks=[WandbCallbackWithStructures(log_model=True, first_observed_fragments=first_observed_fragments)],
    random_state=71
)

model.fit(X_train, y_train, eval_set=[(X_val, y_val)])

# バリデーションセットでの予測
y_pred = model.predict(X_val)

# 結果をテーブルの整理してW&Bサーバーに送る

In [ ]:
def mol_to_pil_image(smiles: str, width: int = 300, height: int = 300) -> "PIL.Image":
    molecule = Chem.MolFromSmiles(smiles)
    Chem.AllChem.Compute2DCoords(molecule)
    Chem.AllChem.GenerateDepictionMatching2DStructure(molecule, molecule)
    pil_image = Chem.Draw.MolToImage(molecule, size=(width, height))
    return wandb.Image(pil_image)

def convert_to_3d_and_optimize(smiles):
    try:
      return wandb.Molecule.from_smiles(smiles)
    except:
      return np.nan

In [ ]:
validation["y_pred"] = y_pred
validation["molecule"] = validation.smiles.progress_apply(convert_to_3d_and_optimize)
validation["molecule_2D"] = validation.smiles.progress_apply(mol_to_pil_image)

In [ ]:
table = wandb.Table(dataframe=validation)
table.add_column("ECFP4", [row.tolist() for row in X_val.values])
run.log({'EDA_Table': table})

In [ ]:
run.finish()